# Lab 1 — LightRAG com OpenSearch e vLLM
## Configuração, Indexação e Primeiras Queries

**Aula 9 | MBA: RAG & CAG Aplicados a Direito e Segurança Pública**

---

### Objetivo
Configurar o LightRAG com:
- **vLLM** como backend LLM (via API compatível com OpenAI)
- **OpenSearch** como storage unificado (vetores + grafo + docs)
- **BAAI/bge-m3** como modelo de embedding local

Indexar o corpus jurídico e executar queries em todos os modos.

---

### ⚠️ Nota sobre o Ambiente Colab
Este lab usa um **servidor vLLM simulado** (via LiteLLM proxy ou OpenAI diretamente) e um **OpenSearch em Docker** para fins didáticos. Em produção, substitua pelas URLs dos servidores on-premise institucionais.

## Passo 1: Instalação das Dependências

In [ ]:
# Passo 1.1 — Instalar LightRAG com suporte a OpenSearch
# O extra [opensearch] inclui o cliente opensearch-py
!pip install "lightrag-hku[opensearch]" --quiet

# Passo 1.2 — Instalar FlagEmbedding para embeddings locais BAAI/bge-m3
!pip install FlagEmbedding --quiet

# Passo 1.3 — Dependências auxiliares
!pip install python-dotenv nest-asyncio aiohttp --quiet

print("✅ Dependências instaladas com sucesso!")

## Passo 2: Subir OpenSearch via Docker (ambiente Colab)

Em produção, este passo é substituído pela conexão ao cluster OpenSearch institucional.

In [ ]:
# Passo 2.1 — Verificar se Docker está disponível no Colab
import subprocess
result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "Docker não disponível — usando OpenSearch remoto")

In [ ]:
# Passo 2.2 — Subir OpenSearch single-node em Docker
# ATENÇÃO: Em Colab, pode ser necessário usar um cluster externo.
# Descomente e execute se Docker estiver disponível.

docker_cmd = """
docker run -d \
  --name opensearch-aula9 \
  -p 9200:9200 -p 9600:9600 \
  -e "discovery.type=single-node" \
  -e "plugins.security.disabled=true" \
  -e "OPENSEARCH_INITIAL_ADMIN_PASSWORD=Admin@1234" \
  opensearchproject/opensearch:2.18.0
"""

print("Comando Docker para iniciar OpenSearch:")
print(docker_cmd)
print()
print("Para executar no terminal:")
print("!" + docker_cmd.strip())

In [ ]:
# Passo 2.3 — Aguardar OpenSearch inicializar e testar conexão
import time
import requests

OPENSEARCH_URL = "http://localhost:9200"  # Altere para URL do cluster em produção

def testar_opensearch(url, tentativas=10, espera=5):
    """Testa se o OpenSearch está pronto para receber conexões."""
    for i in range(tentativas):
        try:
            resp = requests.get(f"{url}/_cluster/health", timeout=5)
            if resp.status_code == 200:
                saude = resp.json()
                print(f"✅ OpenSearch conectado! Status: {saude['status']}")
                print(f"   Nós: {saude['number_of_nodes']} | Shards: {saude['active_shards']}")
                return True
        except Exception as e:
            print(f"Tentativa {i+1}/{tentativas}: aguardando OpenSearch... ({e})")
            time.sleep(espera)
    return False

# Executar teste
# testar_opensearch(OPENSEARCH_URL)  # Descomente após iniciar o Docker

## Passo 3: Configurar Embedding Local com BAAI/bge-m3

In [ ]:
# Passo 3.1 — Carregar o modelo de embedding BAAI/bge-m3
# BAAI/bge-m3 suporta português e produz vetores de dimensão 1024

from FlagEmbedding import FlagModel
import numpy as np

print("Carregando modelo BAAI/bge-m3 (pode demorar na primeira vez)...")

# Carregar modelo de embedding
embedding_model = FlagModel(
    'BAAI/bge-m3',
    query_instruction_for_retrieval="Representar a pergunta para recuperação: ",
    use_fp16=True  # FP16 reduz uso de memória pela metade
)

print("✅ Modelo carregado!")

In [ ]:
# Passo 3.2 — Testar o embedding com texto jurídico

textos_teste = [
    "Organização criminosa com fins de tráfico de entorpecentes",
    "Colaboração premiada no âmbito do Ministério Público Federal",
    "Lavagem de dinheiro via criptomoedas Bitcoin e Monero"
]

# Gerar embeddings
embeddings = embedding_model.encode(textos_teste)

print(f"Dimensão dos embeddings: {embeddings.shape}")
print(f"Tipo: {embeddings.dtype}")
print()

# Calcular similaridade coseno entre os textos
from numpy.linalg import norm

def similaridade_coseno(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("Similaridades entre textos:")
for i in range(len(textos_teste)):
    for j in range(i+1, len(textos_teste)):
        sim = similaridade_coseno(embeddings[i], embeddings[j])
        print(f"  [{i+1}] vs [{j+1}]: {sim:.4f}")

In [ ]:
# Passo 3.3 — Criar função de embedding compatível com LightRAG
# O LightRAG espera uma função async que recebe lista de strings e retorna lista de vetores

import asyncio
from typing import List

async def embedding_func_bge(texts: List[str]) -> np.ndarray:
    """
    Função de embedding assíncrona compatível com LightRAG.
    Usa BAAI/bge-m3 localmente — nenhum dado é enviado para APIs externas.
    
    Args:
        texts: Lista de textos para vetorizar
    
    Returns:
        Array numpy com shape (n_textos, 1024)
    """
    # Executar em thread pool para não bloquear o event loop
    loop = asyncio.get_event_loop()
    embeddings = await loop.run_in_executor(
        None,  # usa ThreadPoolExecutor padrão
        lambda: embedding_model.encode(texts, batch_size=16)
    )
    return embeddings

# Teste rápido da função async
import nest_asyncio
nest_asyncio.apply()  # Necessário no Colab

async def testar_embedding_func():
    resultado = await embedding_func_bge(["Teste de embedding jurídico"])
    print(f"✅ Função de embedding funcionando! Shape: {resultado.shape}")

asyncio.run(testar_embedding_func())

## Passo 4: Configurar vLLM como Backend LLM

In [ ]:
# Passo 4.1 — Configuração do cliente LLM via vLLM
# O vLLM expõe uma API compatível com OpenAI em /v1/chat/completions
# Substitua VLLM_BASE_URL pela URL do seu servidor vLLM on-premise

import os
from lightrag.llm.openai import openai_complete_if_cache

# --- CONFIGURAÇÃO DO VLLM ---
# Em produção: URL do servidor vLLM on-premise
# Ex: "http://vllm-server.intranet.gov.br:8000/v1"
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "meta-llama/Llama-3.1-8B-Instruct")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "token-vllm")  # vLLM aceita qualquer string

# Para fins didáticos no Colab, pode-se usar OpenAI diretamente:
# VLLM_BASE_URL = "https://api.openai.com/v1"
# VLLM_MODEL = "gpt-4o-mini"
# VLLM_API_KEY = os.getenv("OPENAI_API_KEY")

async def llm_model_func(
    prompt: str,
    system_prompt: str = None,
    history_messages: list = [],
    **kwargs
) -> str:
    """
    Função LLM compatível com LightRAG.
    Chama o vLLM via API compatível com OpenAI.
    
    Args:
        prompt: Prompt principal
        system_prompt: Instrução de sistema (opcional)
        history_messages: Histórico de mensagens anteriores
    
    Returns:
        Texto gerado pelo LLM
    """
    return await openai_complete_if_cache(
        model=VLLM_MODEL,
        prompt=prompt,
        system_prompt=system_prompt or (
            "Você é um assistente jurídico especializado em direito penal brasileiro "
            "e segurança pública. Responda sempre em português do Brasil, com precisão "
            "técnica e fundamentação em leis e jurisprudência quando aplicável."
        ),
        history_messages=history_messages,
        api_key=VLLM_API_KEY,
        base_url=VLLM_BASE_URL,
        **kwargs
    )

print(f"✅ Configuração vLLM:")
print(f"   URL: {VLLM_BASE_URL}")
print(f"   Modelo: {VLLM_MODEL}")

In [ ]:
# Passo 4.2 — Testar conectividade com vLLM
import aiohttp
import json

async def testar_vllm():
    """Testa se o servidor vLLM está respondendo."""
    headers = {
        "Authorization": f"Bearer {VLLM_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": VLLM_MODEL,
        "messages": [
            {"role": "user", "content": "Responda apenas: OK"}
        ],
        "max_tokens": 10
    }
    
    try:
        async with aiohttp.ClientSession() as session:
            async with session.post(
                f"{VLLM_BASE_URL}/chat/completions",
                headers=headers,
                json=payload,
                timeout=aiohttp.ClientTimeout(total=30)
            ) as resp:
                if resp.status == 200:
                    data = await resp.json()
                    resposta = data['choices'][0]['message']['content']
                    print(f"✅ vLLM respondendo! Resposta: '{resposta}'")
                else:
                    print(f"⚠️ vLLM retornou status {resp.status}")
    except Exception as e:
        print(f"⚠️ vLLM não acessível: {e}")
        print("   Configure VLLM_BASE_URL com o endereço do seu servidor")

asyncio.run(testar_vllm())

## Passo 5: Inicializar LightRAG com OpenSearch

In [ ]:
# Passo 5.1 — Importações do LightRAG
from lightrag import LightRAG, QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status

# Passo 5.2 — Configurar parâmetros do OpenSearch
OPENSEARCH_CONFIG = {
    "host": "localhost",         # Host do OpenSearch
    "port": 9200,               # Porta padrão
    "use_ssl": False,           # True em produção
    "verify_certs": False,      # True em produção com certificados válidos
    "index_prefix": "aula9",   # Prefixo dos índices criados pelo LightRAG
    # Em produção com autenticação:
    # "http_auth": ("admin", "senha_segura"),
}

# Passo 5.3 — Criar instância do LightRAG
rag = LightRAG(
    # --- LLM ---
    llm_model_func=llm_model_func,
    llm_model_max_token_size=8192,    # Contexto máximo do modelo vLLM
    llm_model_max_async=4,            # Chamadas paralelas ao vLLM
    
    # --- Embeddings ---
    embedding_func=embedding_func_bge,
    embedding_dim=1024,               # Dimensão do BAAI/bge-m3
    embedding_batch_num=16,           # Textos por batch de embedding
    embedding_func_max_async=2,       # Chamadas paralelas de embedding
    
    # --- Storage: OpenSearch ---
    kv_storage="OpensearchKVStorage",
    vector_storage="OpensearchVectorDBStorage",
    graph_storage="OpensearchGraphStorage",
    doc_status_storage="JsonDocStatusStorage",  # Status local
    
    # Diretório de trabalho (para cache e status)
    working_dir="./lightrag_aula9",
    
    # --- Parâmetros de Chunking ---
    chunk_token_size=1200,   # Tokens por chunk (documentos jurídicos são densos)
    chunk_overlap_token_size=100,  # Sobreposição entre chunks
    
    # --- OpenSearch config (passado via addon_params) ---
    addon_params={
        "opensearch_config": OPENSEARCH_CONFIG,
        "language": "Portuguese",  # Idioma para extração de entidades
    }
)

print("✅ LightRAG inicializado com sucesso!")
print(f"   Working dir: ./lightrag_aula9")
print(f"   Embedding dim: 1024 (BAAI/bge-m3)")
print(f"   Storage: OpenSearch")

## Passo 6: Carregar e Indexar o Corpus Jurídico

In [ ]:
# Passo 6.1 — Carregar corpus jurídico
import os

CORPUS_PATH = "../datasets/corpus_juridico.txt"

# Verificar se o arquivo existe
if not os.path.exists(CORPUS_PATH):
    # Criar diretório e baixar do GitHub (para uso standalone)
    os.makedirs("../datasets", exist_ok=True)
    print("⚠️ Arquivo não encontrado. Criando corpus de exemplo...")

with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus_completo = f.read()

# Separar documentos individuais pelo marcador ===DOCUMENTO_XXX===
documentos = []
blocos = corpus_completo.split('===')
for i, bloco in enumerate(blocos):
    if bloco.startswith('DOCUMENTO_') or (i > 0 and blocos[i-1].startswith('DOCUMENTO_')):
        continue
    if bloco.strip() and not bloco.strip().startswith('DOCUMENTO_'):
        documentos.append(bloco.strip())

# Método alternativo: dividir por padrão mais robusto
import re
partes = re.split(r'={3}DOCUMENTO_\d+={3}', corpus_completo)
documentos = [p.strip() for p in partes if p.strip()]

print(f"✅ Corpus carregado: {len(documentos)} documentos")
print(f"   Total de caracteres: {len(corpus_completo):,}")
print()
for i, doc in enumerate(documentos):
    primeira_linha = doc.split('\n')[0][:80]
    print(f"   [{i+1}] {primeira_linha}")

In [ ]:
# Passo 6.2 — Indexar documentos no LightRAG
# Este processo:
# 1. Divide cada documento em chunks
# 2. Para cada chunk, chama o vLLM para extrair entidades e relações
# 3. Gera embeddings com BAAI/bge-m3
# 4. Armazena tudo no OpenSearch

import time

print("🔄 Iniciando indexação do corpus jurídico...")
print("   (Este processo pode levar alguns minutos — o LLM extrai entidades de cada chunk)")
print()

inicio = time.time()

async def indexar_corpus():
    """Indexa todos os documentos do corpus jurídico no LightRAG."""
    await initialize_pipeline_status()
    
    for i, documento in enumerate(documentos):
        print(f"  [{i+1}/{len(documentos)}] Indexando documento...")
        await rag.ainsert(documento)
        print(f"  ✅ Documento {i+1} indexado")
    
    print("\n✅ Indexação concluída!")

# Executar indexação
asyncio.run(indexar_corpus())

tempo_total = time.time() - inicio
print(f"⏱️ Tempo total de indexação: {tempo_total:.1f}s ({tempo_total/len(documentos):.1f}s por documento)")

In [ ]:
# Passo 6.3 — Inspecionar índices criados no OpenSearch
import requests

def listar_indices_opensearch(url_base):
    """Lista todos os índices LightRAG criados no OpenSearch."""
    try:
        resp = requests.get(f"{url_base}/_cat/indices/aula9*?v&s=index")
        if resp.status_code == 200:
            print("📊 Índices LightRAG no OpenSearch:")
            print(resp.text)
        else:
            print(f"Status: {resp.status_code}")
    except Exception as e:
        print(f"Erro ao conectar: {e}")

listar_indices_opensearch(OPENSEARCH_URL)

# Esperamos ver índices como:
# aula9-entities    → entidades extraídas (com vetores kNN)
# aula9-relations   → relações entre entidades
# aula9-docs        → chunks dos documentos originais
# aula9-llm-cache   → cache de chamadas ao vLLM

## Passo 7: Executar Queries nos 4 Modos

In [ ]:
# Passo 7.1 — Função auxiliar para comparar modos

async def comparar_modos(pergunta: str):
    """
    Executa a mesma pergunta em todos os 4 modos do LightRAG
    e exibe os resultados comparativamente.
    
    Args:
        pergunta: Pergunta jurídica em linguagem natural
    """
    modos = ["naive", "local", "global", "hybrid"]
    resultados = {}
    
    print(f"\n{'='*70}")
    print(f"PERGUNTA: {pergunta}")
    print(f"{'='*70}\n")
    
    for modo in modos:
        print(f"\n{'─'*40}")
        print(f"📋 MODO: {modo.upper()}")
        print(f"{'─'*40}")
        
        inicio = time.time()
        try:
            resposta = await rag.aquery(
                pergunta,
                param=QueryParam(mode=modo)
            )
            tempo = time.time() - inicio
            resultados[modo] = resposta
            print(resposta)
            print(f"\n⏱️ Tempo: {tempo:.2f}s")
        except Exception as e:
            print(f"❌ Erro no modo {modo}: {e}")
    
    return resultados

print("✅ Função de comparação de modos carregada")

In [ ]:
# Passo 7.2 — Query 1: Pergunta sobre entidade específica (modo local é melhor)
# Tipo de pergunta: "Quem" / "O que é" / "Quais são as conexões de X"

pergunta_local = "Quais são as técnicas investigativas utilizadas na investigação \
de organizações criminosas e quem as autoriza?"

resultados_q1 = asyncio.run(comparar_modos(pergunta_local))

In [ ]:
# Passo 7.3 — Query 2: Pergunta temática ampla (modo global é melhor)
# Tipo: "Quais os principais temas" / "Que padrões existem" / "Resuma o corpus"

pergunta_global = "Quais são os principais instrumentos jurídicos de \
justiça negociada presentes no corpus e como eles se relacionam entre si?"

resultados_q2 = asyncio.run(comparar_modos(pergunta_global))

In [ ]:
# Passo 7.4 — Query 3: Raciocínio multi-hop (modo hybrid é melhor)
# Tipo: perguntas que conectam múltiplas entidades e temas

pergunta_multihop = "Como o caso de lavagem de dinheiro via criptomoedas se \
relaciona com a lei de organizações criminosas e quais meios de prova foram \
utilizados para identificar o réu?"

resultados_q3 = asyncio.run(comparar_modos(pergunta_multihop))

In [ ]:
# Passo 7.5 — Query com streaming (para interfaces interativas)
# O streaming permite exibir a resposta enquanto ela é gerada

async def query_com_streaming(pergunta: str, modo: str = "hybrid"):
    """Realiza query com resposta em streaming."""
    print(f"\nPergunta ({modo}): {pergunta}\n")
    print("Resposta: ", end="", flush=True)
    
    async for chunk in rag.aquery_stream(
        pergunta,
        param=QueryParam(mode=modo)
    ):
        print(chunk, end="", flush=True)
    
    print("\n")

pergunta_stream = "Explique o standard probatório no direito penal brasileiro \
com base nos documentos disponíveis."

asyncio.run(query_com_streaming(pergunta_stream, modo="hybrid"))

## Passo 8: Inspecionar o Grafo de Conhecimento

In [ ]:
# Passo 8.1 — Exportar o grafo para visualização
# LightRAG pode exportar o grafo em formato GraphML ou JSON

async def exportar_grafo():
    """Exporta o grafo de conhecimento construído pelo LightRAG."""
    # Exportar em formato JSON (lista de nós e arestas)
    grafo_data = await rag.export_graph(file_path="./grafo_juridico.graphml")
    print("✅ Grafo exportado: grafo_juridico.graphml")
    return grafo_data

# asyncio.run(exportar_grafo())

In [ ]:
# Passo 8.2 — Visualizar grafo com NetworkX + Matplotlib

!pip install networkx matplotlib --quiet

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def visualizar_grafo_juridico(graphml_path: str):
    """Visualiza o grafo de conhecimento jurídico."""
    try:
        G = nx.read_graphml(graphml_path)
    except FileNotFoundError:
        # Criar grafo de demonstração para o lab
        print("ℹ️ Usando grafo de demonstração (execute a indexação primeiro)")
        G = nx.DiGraph()
        
        # Entidades
        entidades = [
            ("Lei_12850", {"tipo": "lei"}),
            ("Organização_Criminosa", {"tipo": "conceito"}),
            ("Colaboração_Premiada", {"tipo": "instrumento"}),
            ("Ministério_Público", {"tipo": "instituição"}),
            ("STF", {"tipo": "tribunal"}),
            ("HC_127483", {"tipo": "processo"}),
            ("ANPP", {"tipo": "instrumento"}),
            ("Lei_13964", {"tipo": "lei"}),
            ("Lavagem_Dinheiro", {"tipo": "crime"}),
            ("Criptomoedas", {"tipo": "meio"}),
            ("TRF4", {"tipo": "tribunal"}),
        ]
        G.add_nodes_from(entidades)
        
        # Relações
        relacoes = [
            ("Lei_12850", "Organização_Criminosa", "define"),
            ("Lei_12850", "Colaboração_Premiada", "regulamenta"),
            ("Ministério_Público", "Colaboração_Premiada", "firma"),
            ("STF", "HC_127483", "julgou"),
            ("HC_127483", "Colaboração_Premiada", "trata_de"),
            ("Lei_13964", "ANPP", "criou"),
            ("Ministério_Público", "ANPP", "celebra"),
            ("ANPP", "Colaboração_Premiada", "difere_de"),
            ("Lavagem_Dinheiro", "Criptomoedas", "usa"),
            ("TRF4", "Lavagem_Dinheiro", "condenou"),
        ]
        for s, t, r in relacoes:
            G.add_edge(s, t, label=r)
    
    # Definir cores por tipo de entidade
    cores_tipo = {
        "lei": "#4ECDC4",
        "tribunal": "#45B7D1",
        "instituição": "#96CEB4",
        "instrumento": "#FFEAA7",
        "processo": "#DDA0DD",
        "crime": "#FF6B6B",
        "meio": "#FFA07A",
        "conceito": "#98D8C8",
    }
    
    cores_nos = []
    for no in G.nodes():
        tipo = G.nodes[no].get('tipo', 'conceito')
        cores_nos.append(cores_tipo.get(tipo, '#CCCCCC'))
    
    # Layout do grafo
    plt.figure(figsize=(16, 10))
    pos = nx.spring_layout(G, k=2.5, iterations=50, seed=42)
    
    # Desenhar grafo
    nx.draw_networkx_nodes(G, pos, node_color=cores_nos, node_size=2000, alpha=0.9)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
    nx.draw_networkx_edges(G, pos, edge_color='#666666', arrows=True,
                          arrowsize=20, width=1.5, alpha=0.7,
                          connectionstyle='arc3,rad=0.1')
    
    # Labels das arestas
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, alpha=0.8)
    
    # Legenda
    patches = [mpatches.Patch(color=c, label=t.capitalize()) 
               for t, c in cores_tipo.items()]
    plt.legend(handles=patches, loc='upper left', fontsize=8)
    
    plt.title("Grafo de Conhecimento Jurídico — LightRAG + OpenSearch",
              fontsize=14, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('grafo_juridico.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Grafo salvo: grafo_juridico.png")
    print(f"   Nós (entidades): {G.number_of_nodes()}")
    print(f"   Arestas (relações): {G.number_of_edges()}")

visualizar_grafo_juridico("./grafo_juridico.graphml")

In [ ]:
# Passo 8.3 — Analisar métricas do grafo

def analisar_metricas_grafo(G: nx.DiGraph):
    """Calcula métricas de centralidade e estrutura do grafo."""
    print("📊 MÉTRICAS DO GRAFO DE CONHECIMENTO JURÍDICO")
    print("=" * 50)
    print(f"Total de entidades: {G.number_of_nodes()}")
    print(f"Total de relações: {G.number_of_edges()}")
    print(f"Densidade: {nx.density(G):.4f}")
    print()
    
    # Grau de entrada (quem é mais referenciado)
    in_degree = dict(G.in_degree())
    top_referenciados = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:5]
    print("🏆 Entidades mais referenciadas (in-degree):")
    for entidade, grau in top_referenciados:
        print(f"   {entidade}: {grau} referências")
    print()
    
    # PageRank (importância no grafo)
    if G.number_of_edges() > 0:
        pagerank = nx.pagerank(G)
        top_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]
        print("⭐ Entidades mais importantes (PageRank):")
        for entidade, score in top_pr:
            print(f"   {entidade}: {score:.4f}")

# Usar o grafo de demonstração criado no passo anterior
G_demo = nx.DiGraph()
entidades_demo = [
    "Lei_12850", "Organização_Criminosa", "Colaboração_Premiada",
    "Ministério_Público", "STF", "HC_127483", "ANPP", "Lei_13964",
    "Lavagem_Dinheiro", "Criptomoedas", "TRF4"
]
relacoes_demo = [
    ("Lei_12850", "Organização_Criminosa"), ("Lei_12850", "Colaboração_Premiada"),
    ("Ministério_Público", "Colaboração_Premiada"), ("STF", "HC_127483"),
    ("HC_127483", "Colaboração_Premiada"), ("Lei_13964", "ANPP"),
    ("Ministério_Público", "ANPP"), ("ANPP", "Colaboração_Premiada"),
    ("Lavagem_Dinheiro", "Criptomoedas"), ("TRF4", "Lavagem_Dinheiro"),
]
G_demo.add_nodes_from(entidades_demo)
G_demo.add_edges_from(relacoes_demo)

analisar_metricas_grafo(G_demo)

## Passo 9: Atualização Incremental do Corpus

Uma das vantagens do LightRAG é o suporte a inserção incremental — novos documentos são integrados ao grafo existente sem reprocessar tudo.

In [ ]:
# Passo 9.1 — Inserir novo documento no grafo já existente

novo_documento = """
TÍTULO: Operação Hydra — Relatório Final
TIPO: Relatório de Inteligência Policial
DATA: Janeiro de 2025
ÓRGÃO: Polícia Federal — Delegacia de Repressão a Crimes Financeiros

A Operação Hydra identificou rede de lavagem de dinheiro com conexão direta
à organização investigada na Operação Integração. O mesmo grupo, liderado por
Rodrigo Menezes, utilizava as criptomoedas identificadas no caso Criptomarcus
como veículo de lavagem para recursos do tráfico.

A colaboração premiada firmada por um dos integrantes permitiu identificar
cinco novas empresas de fachada, conectadas ao esquema financeiro.

ENTIDADES: [Operação_Hydra, Rodrigo_Menezes, Polícia_Federal, Lavagem_Dinheiro,
            Operação_Integração, Criptomoedas, Colaboração_Premiada, Empresas_Fachada]
RELAÇÕES: [Hydra_conecta -> Integração, Menezes_lidera -> Rede_Lavagem,
           Colaboração_revela -> Empresas_Fachada]
"""

async def inserir_novo_documento():
    print("🔄 Inserindo novo documento no grafo existente...")
    await rag.ainsert(novo_documento)
    print("✅ Novo documento integrado!")
    print("   O grafo foi atualizado sem reprocessar os documentos anteriores.")

asyncio.run(inserir_novo_documento())

In [ ]:
# Passo 9.2 — Verificar que as conexões cross-documento funcionam

async def testar_conexao_cross_documento():
    """Testa se o novo documento foi integrado ao grafo corretamente."""
    pergunta = "Qual a relação entre a Operação Hydra e a Operação Integração?"
    
    print(f"Pergunta: {pergunta}\n")
    
    resposta = await rag.aquery(
        pergunta,
        param=QueryParam(mode="local")
    )
    
    print("Resposta (modo local):")
    print(resposta)

asyncio.run(testar_conexao_cross_documento())

## Resumo do Lab 1

Neste laboratório você:

1. ✅ Instalou o LightRAG com suporte a OpenSearch
2. ✅ Configurou embeddings locais com BAAI/bge-m3 (sem envio de dados para APIs externas)
3. ✅ Integrou vLLM como backend LLM via API compatível com OpenAI
4. ✅ Indexou corpus jurídico com extração automática de entidades e relações
5. ✅ Executou queries em 4 modos: naive, local, global, hybrid
6. ✅ Visualizou o grafo de conhecimento com NetworkX
7. ✅ Realizou atualização incremental do grafo com novo documento

**Próximo Lab:** Lab 2 — Queries Avançadas e Investigação de Redes Criminosas com Graph RAG